# Tutorial 4: Molecule (Drug) Embeddings

This notebook covers generating embeddings for **small molecules / drugs**
with `embpy`. Molecules are represented as **SMILES** strings.

## Available molecule embedding models

### Transformer-based models

| Model key | Architecture | Notes |
|---|---|---|
| `chemberta2MTR` | ChemBERTa-2 (MTR) | Transformer, multi-task regression head |
| `chemberta2MLM` | ChemBERTa-2 (MLM) | Transformer, masked-language-model head |
| `molformer_base` | MolFormer | Large-scale molecular transformer |

### GNN-based models

| Model key | Architecture | Notes |
|---|---|---|
| `minimol` | MiniMol (GINE) | 10M-param message-passing GNN, 512-dim fingerprints. `pip install minimol` |
| `mhg_gnn` | MHG-GNN (GIN + MHG decoder) | Hypergraph autoencoder, can **decode** embeddings back to SMILES. Install from HuggingFace. |
| `mole` | MolE (DeBERTa-based Graph Transformer) | Pre-trained on 842M molecules. **Weights not yet publicly released.** |

### RDKit fingerprints (non-learned baselines)

| Model key | Type | Notes |
|---|---|---|
| `rdkit_fp` | RDKit topological | Binary (0/1), 2048-dim |
| `morgan_fp` / `morgan_count_fp` | Morgan / ECFP | Binary or count-based |
| `maccs_fp` | MACCS keys | Binary, 167-dim |
| `atom_pair_fp` / `atom_pair_count_fp` | Atom pair | Binary or count-based |
| `torsion_fp` / `torsion_count_fp` | Topological torsion | Binary or count-based |

If you have a drug *name* instead of a SMILES, use the `DrugResolver`
(see [Tutorial 1](01_identifiers_and_preprocessing.ipynb)) to convert it first,
or let `embed_molecule` handle it automatically.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Resolve drug names to SMILES

Most embedding models expect canonical SMILES. The `DrugResolver` maps
common drug names to SMILES via PubChem.

In [ ]:
from embpy.resources.drug_resolver import DrugResolver

drug_resolver = DrugResolver()

drugs = {
    "aspirin":    drug_resolver.name_to_smiles("aspirin"),
    "ibuprofen":  drug_resolver.name_to_smiles("ibuprofen"),
    "caffeine":   drug_resolver.name_to_smiles("caffeine"),
    "paclitaxel": drug_resolver.name_to_smiles("paclitaxel"),
}

for name, smi in drugs.items():
    print(f"  {name:12s} → {smi}")

## 2. Embed a single molecule

In [ ]:
MODEL = "chemberta2MTR"

aspirin_smi = drugs["aspirin"]
print(f"Aspirin SMILES: {aspirin_smi}")

emb = embedder.embed_molecule(
    identifier=aspirin_smi,
    model=MODEL,
    pooling_strategy="mean",
)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding dtype: {emb.dtype}")

## 3. Batch molecule embedding

In [ ]:
smiles_list = [smi for smi in drugs.values() if smi is not None]
names_list = [n for n, s in drugs.items() if s is not None]

embeddings = embedder.embed_molecules_batch(
    identifiers=smiles_list,
    model=MODEL,
    pooling_strategy="mean",
)

for name, emb in zip(names_list, embeddings):
    if emb is not None:
        print(f"  {name:12s} → dim {emb.shape[0]}")
    else:
        print(f"  {name:12s} → FAILED")

## 4. Comparing transformer-based molecule models

Different models produce embeddings in different spaces and dimensions.

In [ ]:
import numpy as np

test_smi = drugs["caffeine"]

for m in ["chemberta2MTR", "chemberta2MLM", "molformer_base"]:
    try:
        e = embedder.embed_molecule(test_smi, model=m, pooling_strategy="mean")
        print(f"  {m:20s} → dim {e.shape[0]:,}, L2 = {np.linalg.norm(e):.2f}")
    except Exception as exc:
        print(f"  {m:20s} → {exc}")

## 5. Similarity between molecules

Embedding-based cosine similarity can capture structural and functional
relatedness between drugs.

In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Embed two structurally related NSAIDs (aspirin & ibuprofen)
# and one very different molecule (caffeine)
embs = {}
for name in ["aspirin", "ibuprofen", "caffeine"]:
    smi = drugs[name]
    if smi:
        embs[name] = embedder.embed_molecule(smi, model=MODEL, pooling_strategy="mean")

if len(embs) == 3:
    print(f"aspirin  vs ibuprofen: {cosine_sim(embs['aspirin'], embs['ibuprofen']):.4f}")
    print(f"aspirin  vs caffeine:  {cosine_sim(embs['aspirin'], embs['caffeine']):.4f}")
    print(f"ibuprofen vs caffeine: {cosine_sim(embs['ibuprofen'], embs['caffeine']):.4f}")

## 6. Reverse lookup: SMILES → drug name

In [ ]:
# What is this molecule?
unknown_smi = "CC(=O)Oc1ccccc1C(O)=O"

primary = drug_resolver.smiles_to_primary_name(unknown_smi)
all_names = drug_resolver.smiles_to_names(unknown_smi, top_k=5)

print(f"Primary name: {primary}")
print(f"All names:    {all_names}")

## 8. GNN-based molecule models

`embpy` also supports **graph neural network** models that operate on the
molecular graph structure (atoms as nodes, bonds as edges), rather than
treating SMILES as a text sequence.

> **Note**: Each GNN model requires its own optional dependency. See the
> model table at the top for installation instructions.

### 8.1 MiniMol — Message-Passing GNN (GINE)

MiniMol is a 10M-parameter pre-trained **GINE** (GIN with edge features)
model trained on 6M molecules across 3,300+ bio/quantum tasks. It produces
fixed **512-dimensional** fingerprints.

```bash
pip install minimol
```

```python
# Using MiniMol through the BioEmbedder
emb_minimol = embedder.embed_molecule(
    identifier=drugs["aspirin"],
    model="minimol",
)
print(f"MiniMol embedding shape: {emb_minimol.shape}")
```

Or use the wrapper directly:

```python
from embpy.models.molecule_models import MiniMolWrapper

wrapper = MiniMolWrapper(batch_size=64)
wrapper.load(embedder.device)

emb = wrapper.embed("CCO")
print(f"Ethanol embedding: shape={emb.shape}, dtype={emb.dtype}")
# → shape=(512,), dtype=float32

# Batch embedding
batch_embs = wrapper.embed_batch(["CCO", "c1ccccc1", "CC(=O)O"])
print(f"Batch: {len(batch_embs)} embeddings of dim {batch_embs[0].shape[0]}")
```

### 8.2 MHG-GNN — Molecular Hypergraph Grammar + GNN

MHG-GNN is a **GIN-based autoencoder** pre-trained on ~1.34M PubChem molecules.
Its unique feature is that the decoder can reconstruct **structurally valid SMILES**
from latent vectors.

```bash
pip install torch-geometric huggingface-hub
pip install git+https://github.com/GT4SD/mhg-gnn.git
```

```python
from embpy.models.molecule_models import MHGGNNWrapper

wrapper = MHGGNNWrapper()
wrapper.load(embedder.device)

# Encode
emb = wrapper.embed("CCO")
print(f"Latent embedding: shape={emb.shape}")

# Batch encode
embs = wrapper.embed_batch(["CCO", "c1ccccc1"])
print(f"Batch: {len(embs)} embeddings")

# Decode — reconstruct SMILES from latent vectors
decoded = wrapper.decode(embs)
print(f"Decoded SMILES: {decoded}")
```

> **Unique capability**: MHG-GNN can **decode** latent vectors back to valid
> SMILES strings, enabling molecular generation and interpolation in
> the latent space.

### 8.3 MolE — Graph Transformer (DeBERTa-based)

MolE is a **DeBERTa-based graph transformer** pre-trained on 842M molecules from
PubChem and ZINC. It uses a CLS-token embedding for molecular representation.

> **Note**: As of writing, pretrained checkpoints for MolE have not yet been
> publicly released by the authors. You will need to obtain a checkpoint file
> before using this wrapper.

```python
from embpy.models.molecule_models import MolEWrapper

wrapper = MolEWrapper(checkpoint_path="/path/to/mole_checkpoint.pt")
wrapper.load(embedder.device)

emb = wrapper.embed("CCO")
print(f"MolE embedding: shape={emb.shape}")
```

## 9. RDKit fingerprints (non-learned baselines)

`embpy` wraps several popular **RDKit fingerprint** types through the
`RDKitWrapper`. These are deterministic (no model loading needed),
fast, and useful as baselines or for traditional ML pipelines.

```python
from embpy.models.molecule_models import RDKitWrapper

smi = "CC(=O)Oc1ccccc1C(O)=O"  # aspirin

# Binary Morgan fingerprint (ECFP4-like, 2048-dim)
rw = RDKitWrapper(fp_type="morgan_fp")
rw.load(embedder.device)
fp_binary = rw.embed(smi)
print(f"Morgan (binary): shape={fp_binary.shape}, unique values={set(fp_binary.flatten())}")

# Count-based Morgan fingerprint
rw_count = RDKitWrapper(fp_type="morgan_count_fp")
rw_count.load(embedder.device)
fp_count = rw_count.embed(smi)
print(f"Morgan (count): shape={fp_count.shape}, max count={fp_count.max():.0f}")

# MACCS keys (167-dim)
rw_maccs = RDKitWrapper(fp_type="maccs_fp")
rw_maccs.load(embedder.device)
fp_maccs = rw_maccs.embed(smi)
print(f"MACCS: shape={fp_maccs.shape}")
```

Available fingerprint types:
- `rdkit_fp` — RDKit topological (binary, 2048-dim)
- `morgan_fp` / `morgan_count_fp` — Morgan/ECFP (binary or count, 2048-dim)
- `maccs_fp` — MACCS keys (binary, 167-dim)
- `atom_pair_fp` / `atom_pair_count_fp` — Atom pair (binary or count, 2048-dim)
- `torsion_fp` / `torsion_count_fp` — Topological torsion (binary or count, 2048-dim)

## 10. Building an embedding matrix from an AnnData

If you have a perturbation-screen `AnnData` whose `.obs` contains a
column with drug names or SMILES, use
`PerturbationProcessor.build_molecule_embedding_matrix` to embed them
all at once. It resolves names → SMILES automatically, embeds, and
stores the result in `.obsm`.

This works with **any** of the models above — transformer, GNN, or RDKit.

In [ ]:
import anndata as ad
import pandas as pd
from embpy.pp.basic import PerturbationProcessor

pp = PerturbationProcessor(embedder=embedder)

# Simulate a screen AnnData with drug metadata
screen_obs = pd.DataFrame({
    "treatment":  ["aspirin", "ibuprofen", "caffeine", "paclitaxel"],
    "dose_uM":    [10.0, 5.0, 1.0, 0.1],
    "cell_line":  ["A549", "A549", "HeLa", "HeLa"],
})
adata_screen = ad.AnnData(obs=screen_obs)

# Embed – just point at the .obs column with drug names
adata_emb = pp.build_molecule_embedding_matrix(
    adata=adata_screen,
    column="treatment",        # which .obs column has drug identifiers
    id_type="drug_name",       # they are common names, not SMILES
    model="chemberta2MTR",
    obsm_key="X_chemberta",
)

print("Result AnnData (original metadata preserved):")
print(adata_emb.obs)
print(f"\nEmbedding matrix: {adata_emb.obsm['X_chemberta'].shape}")
print(f"Successful: {adata_emb.obs['embedded'].sum()} / {len(adata_emb)}")

## Summary

### Resolve & embed

| What | How |
|---|---|
| Drug name → SMILES | `DrugResolver().name_to_smiles("aspirin")` |
| SMILES → drug name | `DrugResolver().smiles_to_names(smiles)` |
| Single molecule embedding | `embed_molecule(smiles, model)` |
| Batch molecule embedding | `embed_molecules_batch(smiles_list, model)` |
| Embed from AnnData column | `pp.build_molecule_embedding_matrix(adata=..., column="treatment")` |
| Embed from a list | `pp.build_molecule_embedding_matrix(identifiers=[...])` |

### Available model categories

| Category | Models | Key strengths |
|---|---|---|
| **Transformer** | ChemBERTa-2 (MTR/MLM), MolFormer | Large-scale pretraining on SMILES strings |
| **Message-Passing GNN** | MiniMol (GINE) | Lightweight, 512-dim graph-level fingerprints |
| **Graph Autoencoder** | MHG-GNN | Encode **and decode** — reconstruct valid SMILES from latent vectors |
| **Graph Transformer** | MolE | DeBERTa-based, pre-trained on 842M molecules |
| **RDKit fingerprints** | Morgan, MACCS, atom-pair, torsion, RDKit-FP | Deterministic, fast, binary or count-based |

Next: [Tutorial 5 – Text Embeddings](05_text_embeddings.ipynb)